In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
base_path = "file:/Workspace/Users/azuser7222_mml.local@karthikirisoutlook.onmicrosoft.com"

doc_df = spark.read.csv(f"{base_path}/doctors.csv", header=True, inferSchema=True)

visit_df = spark.read.csv(f"{base_path}/visits.csv", header=True, inferSchema=True)



In [0]:
doc_df.printSchema()

visit_df.printSchema()



root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- consultation_f: integer (nullable = true)

root
 |-- visit_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: integer (nullable = true)
 |-- payment_: string (nullable = true)



In [0]:
doc_df.count()

8

In [0]:
visit_df.count()

10

In [0]:
doc_df.filter(doc_df.city=="Hyderabad").show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
doc_df.filter(doc_df.specialization=="Cardiology").show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
doc_df.filter(doc_df.experience_years >10).show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
visit_df.filter(visit_df.bill_amount >5000).show()

+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|    Paid|
+--------+------------+---------+----------+-------------+-----------+--------+



In [0]:
visit_df.filter(visit_df.payment_ =="Pending").show()

+--------+------------+---------+----------+------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+------------+-----------+--------+
|   V1003|  Amit Kumar|     D103|2026-01-11|Skin Allergy|       2000| Pending|
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL| Pending|
+--------+------------+---------+----------+------------+-----------+--------+



In [0]:
visit_df.filter(visit_df.payment_ =="Paid").show()

+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|    Paid|
+--------+------------+---------+----------+-------------+-----------+--------+



In [0]:
doc_df.groupBy("specialization").agg(
    avg("consultation_f").alias("average_fee")
).show()

+--------------+-----------+
|specialization|average_fee|
+--------------+-----------+
|   Orthopedics|     2500.0|
|    Cardiology|     2250.0|
|    Pediatrics|     1200.0|
|   Dermatology|     1050.0|
|     Neurology|     1900.0|
+--------------+-----------+



In [0]:
doc_df.groupBy("specialization").agg(
    max("consultation_f").alias("maximum_fee")
).show()

+--------------+-----------+
|specialization|maximum_fee|
+--------------+-----------+
|   Orthopedics|       2500|
|    Cardiology|       3000|
|    Pediatrics|       1200|
|   Dermatology|       1100|
|     Neurology|       2000|
+--------------+-----------+



In [0]:
doc_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|    Delhi|    1|
|  Chennai|    1|
|    Kochi|    1|
|Hyderabad|    2|
|Bangalore|    1|
|     Pune|    1|
|   Mumbai|    1|
+---------+-----+



In [0]:
doc_df.groupBy("specialization").count().show()

+--------------+-----+
|specialization|count|
+--------------+-----+
|   Orthopedics|    1|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Dermatology|    2|
|     Neurology|    2|
+--------------+-----+



In [0]:
visit_df.agg(sum("bill_amount").alias("total_bill")).show()

+----------+
|total_bill|
+----------+
|     46500|
+----------+



In [0]:
visit_df.agg(avg("bill_amount").alias("average_bill")).show()

+-----------------+
|     average_bill|
+-----------------+
|5166.666666666667|
+-----------------+



In [0]:
visit_df.agg(max("bill_amount").alias("Max_bill")).show()

+--------+
|Max_bill|
+--------+
|   12000|
+--------+



In [0]:
visit_df.agg(min("bill_amount").alias("Min_bill")).show()

+--------+
|Min_bill|
+--------+
|    1500|
+--------+



In [0]:
doc_df.orderBy(col("consultation_f").desc()).show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|          1800|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|          1100|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
visit_df.orderBy(col("bill_amount").desc()).show()

+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|    Paid|
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid|
|   V1008|  Meera Nair|     D103|2026-01

In [0]:
visit_df.filter(col("bill_amount").isNull()).show()

+--------+------------+---------+----------+------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+------------+-----------+--------+
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL| Pending|
+--------+------------+---------+----------+------------+-----------+--------+



In [0]:
visit_df = visit_df.fillna(
    {
        "bill_amount": 0
    }
)

visit_df.show()

+--------+------------+---------+----------+-------------+-----------+--------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|
+--------+------------+---------+----------+-------------+-----------+--------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|
|   V1008|  Meera Nair|     D103|2026-01-13| Skin Allergy|          0| Pending|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|    Paid|
|   V1010| Nisha Reddy|     D101|2026-01

In [0]:
visit_df = visit_df.withColumn(
    "tax",
    col("bill_amount") * 0.05
)

visit_df.show()

+--------+------------+---------+----------+-------------+-----------+--------+-----+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|  tax|
+--------+------------+---------+----------+-------------+-----------+--------+-----+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|250.0|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|175.0|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|100.0|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|600.0|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid| 75.0|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|350.0|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|200.0|
|   V1008|  Meera Nair|     D103|2026-01-13| Skin Allergy|          0| Pending|  0.0|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back P

In [0]:
visit_df = visit_df.withColumn(
    "final_bill",
    col("bill_amount") + col("tax")
)

visit_df.show()

+--------+------------+---------+----------+-------------+-----------+--------+-----+----------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+--------+------------+---------+----------+-------------+-----------+--------+-----+----------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|    Paid|600.0|   12600.0|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|    Paid| 75.0|    1575.0|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|    Paid|350.0|    7350.0|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|    Paid|200.0|    4200.0|
|   V1008|  Meera Nair|     D1

In [0]:
doctor_visit_df = doc_df.join(
    visit_df,"doctor_id","inner"
)

doctor_visit_df.show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   

In [0]:
doctor_left_df = doc_df.join(
    visit_df,"doctor_id","left"
)

doctor_left_df.show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|   V1010| Nisha Reddy|2026-01-14|Heart Checkup|       5500|    Paid|275.0|    5775.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1008|  Meera Nair|2026-01-13| Skin Allergy|          0| Pending|  0.0|       0.0|
|   

In [0]:
doctor_right_df = doc_df.join(
    visit_df,
    "doctor_id",
    "right"
)

doctor_right_df.show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|    Paid|250.0|    5250.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   

In [0]:
doctor_full_df = doc_df.join(
    visit_df,
    "doctor_id",
    "full"
)

doctor_full_df.show()


+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D120|       NULL|          NULL|     NULL|            NULL|          NULL|   V1007| Arjun Verma|2026-01-13|     Migraine|       4000|    Paid|200.0|    4200.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000| Pending|100.0|    2100.0|
|   

In [0]:
visit_df.join(doc_df,"doctor_id","left_anti").show()

+---------+--------+------------+----------+---------+-----------+--------+-----+----------+
|doctor_id|visit_id|patient_name|visit_date|diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+--------+------------+----------+---------+-----------+--------+-----+----------+
|     D120|   V1007| Arjun Verma|2026-01-13| Migraine|       4000|    Paid|200.0|    4200.0|
+---------+--------+------------+----------+---------+-----------+--------+-----+----------+



In [0]:
doc_df.join(visit_df,"doctor_id","left_anti").show()

+---------+-----------+--------------+-----+----------------+--------------+
|doctor_id|doctor_name|specialization| city|experience_years|consultation_f|
+---------+-----------+--------------+-----+----------------+--------------+
|     D107| Dr. Farhan|     Neurology| Pune|               5|          1800|
|     D108|  Dr. Nisha|   Dermatology|Kochi|               9|          1100|
+---------+-----------+--------------+-----+----------------+--------------+



In [0]:
doctor_visit_df.groupBy("doctor_name").agg(
    count("visit_id").alias("total_visits")).show()

+-----------+------------+
|doctor_name|total_visits|
+-----------+------------+
| Dr. Ramesh|           2|
|  Dr. Meera|           1|
| Dr. Suresh|           2|
|  Dr. Anita|           2|
|  Dr. Kiran|           1|
|  Dr. Priya|           1|
+-----------+------------+



In [0]:
doctor_visit_df.groupBy("doctor_name").agg(
    sum("bill_amount").alias("total_revenue")).show()

+-----------+-------------+
|doctor_name|total_revenue|
+-----------+-------------+
| Dr. Ramesh|        10500|
|  Dr. Meera|         1500|
| Dr. Suresh|        18000|
|  Dr. Anita|         2000|
|  Dr. Kiran|         7000|
|  Dr. Priya|         3500|
+-----------+-------------+



In [0]:
doctor_visit_df.groupBy(
    "doctor_name").agg(
    sum("bill_amount").alias("total_revenue")).orderBy(
    col("total_revenue").desc()).limit(1).show()

+-----------+-------------+
|doctor_name|total_revenue|
+-----------+-------------+
| Dr. Suresh|        18000|
+-----------+-------------+



In [0]:
doctor_visit_df.groupBy(
    "specialization").agg(
    sum("bill_amount").alias("total_revenue")).orderBy(
    col("total_revenue").desc()).limit(1).show()

+--------------+-------------+
|specialization|total_revenue|
+--------------+-------------+
|   Orthopedics|        18000|
+--------------+-------------+



In [0]:
doctor_visit_df.groupBy(
    "specialization").agg(
    avg("bill_amount").alias("average_revenue")).show()

+--------------+-----------------+
|specialization|  average_revenue|
+--------------+-----------------+
|   Orthopedics|           9000.0|
|    Cardiology|5833.333333333333|
|    Pediatrics|           1500.0|
|   Dermatology|           1000.0|
|     Neurology|           3500.0|
+--------------+-----------------+



In [0]:
doctor_visit_df.groupBy("city").agg(
    sum("bill_amount").alias("total_revenue")
).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|    Delhi|         1500|
|  Chennai|         2000|
|Hyderabad|        17500|
|Bangalore|         3500|
|   Mumbai|        18000|
+---------+-------------+



In [0]:
doctor_visit_df.groupBy(
    "doctor_id",
    "doctor_name").agg(
    count("visit_id").alias("visit_count")).show()

+---------+-----------+-----------+
|doctor_id|doctor_name|visit_count|
+---------+-----------+-----------+
|     D102|  Dr. Priya|          1|
|     D105|  Dr. Meera|          1|
|     D104| Dr. Suresh|          2|
|     D103|  Dr. Anita|          2|
|     D101| Dr. Ramesh|          2|
|     D106|  Dr. Kiran|          1|
+---------+-----------+-----------+



In [0]:
doctor_visit_df.groupBy(
    "doctor_name").agg(
    sum("bill_amount").alias("total_revenue")).orderBy(
    col("total_revenue").desc()
).limit(3).show()

+-----------+-------------+
|doctor_name|total_revenue|
+-----------+-------------+
| Dr. Suresh|        18000|
| Dr. Ramesh|        10500|
|  Dr. Kiran|         7000|
+-----------+-------------+



In [0]:
doctor_visit_df.groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city").agg(
    count("visit_id").alias("total_visits"),
    sum("bill_amount").alias("total_revenue"),
    avg("bill_amount").alias("average_bill"),
    max("bill_amount").alias("highest_bill")).orderBy(
    col("total_revenue").desc()
).show()

+---------+-----------+--------------+---------+------------+-------------+------------+------------+
|doctor_id|doctor_name|specialization|     city|total_visits|total_revenue|average_bill|highest_bill|
+---------+-----------+--------------+---------+------------+-------------+------------+------------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|           2|        18000|      9000.0|       12000|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|           2|        10500|      5250.0|        5500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|           1|         7000|      7000.0|        7000|
|     D102|  Dr. Priya|     Neurology|Bangalore|           1|         3500|      3500.0|        3500|
|     D103|  Dr. Anita|   Dermatology|  Chennai|           2|         2000|      1000.0|        2000|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|           1|         1500|      1500.0|        1500|
+---------+-----------+--------------+---------+------------+-------------+-------

In [0]:
hosp_df = spark.read.option("multiLine", "true").json(f"{base_path}/hospital_config.json")

In [0]:
hosp_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)



In [0]:
flat_df = hosp_df.select(
    "hospital_id",
    "hospital_name",
    "city",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    "services"
)

flat_df.show(truncate=False)

+-----------+----------------+---------+----------+----------------+------------------------------------+
|hospital_id|hospital_name   |city     |phone     |email           |services                            |
+-----------+----------------+---------+----------+----------------+------------------------------------+
|H101       |Apollo Hospital |Hyderabad|9876500011|apollo@mail.com |[Cardiology, Neurology, Dermatology]|
|H102       |Yashoda Hospital|Hyderabad|NULL      |yashoda@mail.com|[Cardiology, Orthopedics]           |
|H103       |Care Hospital   |Bangalore|9876500013|NULL            |[Neurology, Pediatrics]             |
+-----------+----------------+---------+----------+----------------+------------------------------------+



In [0]:
flat_df.filter(col("phone").isNull()).show()

+-----------+----------------+---------+-----+----------------+--------------------+
|hospital_id|   hospital_name|     city|phone|           email|            services|
+-----------+----------------+---------+-----+----------------+--------------------+
|       H102|Yashoda Hospital|Hyderabad| NULL|yashoda@mail.com|[Cardiology, Orth...|
+-----------+----------------+---------+-----+----------------+--------------------+



In [0]:
flat_df.filter(col("email").isNull()).show()

+-----------+-------------+---------+----------+-----+--------------------+
|hospital_id|hospital_name|     city|     phone|email|            services|
+-----------+-------------+---------+----------+-----+--------------------+
|       H103|Care Hospital|Bangalore|9876500013| NULL|[Neurology, Pedia...|
+-----------+-------------+---------+----------+-----+--------------------+



In [0]:
flat_df = flat_df.fillna(
    {
        "phone": "Not Provided",
        "email": "Not Provided"
    }
)

flat_df.show()

+-----------+----------------+---------+------------+----------------+--------------------+
|hospital_id|   hospital_name|     city|       phone|           email|            services|
+-----------+----------------+---------+------------+----------------+--------------------+
|       H101| Apollo Hospital|Hyderabad|  9876500011| apollo@mail.com|[Cardiology, Neur...|
|       H102|Yashoda Hospital|Hyderabad|Not Provided|yashoda@mail.com|[Cardiology, Orth...|
|       H103|   Care Hospital|Bangalore|  9876500013|    Not Provided|[Neurology, Pedia...|
+-----------+----------------+---------+------------+----------------+--------------------+



In [0]:
flat_df.select("hospital_name","city").show()

+----------------+---------+
|   hospital_name|     city|
+----------------+---------+
| Apollo Hospital|Hyderabad|
|Yashoda Hospital|Hyderabad|
|   Care Hospital|Bangalore|
+----------------+---------+



In [0]:
flat_df.select("hospital_name","phone").show()

+----------------+------------+
|   hospital_name|       phone|
+----------------+------------+
| Apollo Hospital|  9876500011|
|Yashoda Hospital|Not Provided|
|   Care Hospital|  9876500013|
+----------------+------------+



In [0]:
services_df = flat_df.withColumn(
    "service",
    explode("services")
)

services_df.show()

+-----------+----------------+---------+------------+----------------+--------------------+-----------+
|hospital_id|   hospital_name|     city|       phone|           email|            services|    service|
+-----------+----------------+---------+------------+----------------+--------------------+-----------+
|       H101| Apollo Hospital|Hyderabad|  9876500011| apollo@mail.com|[Cardiology, Neur...| Cardiology|
|       H101| Apollo Hospital|Hyderabad|  9876500011| apollo@mail.com|[Cardiology, Neur...|  Neurology|
|       H101| Apollo Hospital|Hyderabad|  9876500011| apollo@mail.com|[Cardiology, Neur...|Dermatology|
|       H102|Yashoda Hospital|Hyderabad|Not Provided|yashoda@mail.com|[Cardiology, Orth...| Cardiology|
|       H102|Yashoda Hospital|Hyderabad|Not Provided|yashoda@mail.com|[Cardiology, Orth...|Orthopedics|
|       H103|   Care Hospital|Bangalore|  9876500013|    Not Provided|[Neurology, Pedia...|  Neurology|
|       H103|   Care Hospital|Bangalore|  9876500013|    Not Pro

In [0]:
services_df.select("hospital_name","service").show()

+----------------+-----------+
|   hospital_name|    service|
+----------------+-----------+
| Apollo Hospital| Cardiology|
| Apollo Hospital|  Neurology|
| Apollo Hospital|Dermatology|
|Yashoda Hospital| Cardiology|
|Yashoda Hospital|Orthopedics|
|   Care Hospital|  Neurology|
|   Care Hospital| Pediatrics|
+----------------+-----------+



In [0]:
flat_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    2|
|Bangalore|    1|
+---------+-----+



In [0]:
services_df.groupBy("service").count().show()

+-----------+-----+
|    service|count|
+-----------+-----+
|Orthopedics|    1|
| Cardiology|    2|
| Pediatrics|    1|
|Dermatology|    1|
|  Neurology|    2|
+-----------+-----+



In [0]:
services_df.filter(col("service") == "Cardiology").show()

+-----------+----------------+---------+------------+----------------+--------------------+----------+
|hospital_id|   hospital_name|     city|       phone|           email|            services|   service|
+-----------+----------------+---------+------------+----------------+--------------------+----------+
|       H101| Apollo Hospital|Hyderabad|  9876500011| apollo@mail.com|[Cardiology, Neur...|Cardiology|
|       H102|Yashoda Hospital|Hyderabad|Not Provided|yashoda@mail.com|[Cardiology, Orth...|Cardiology|
+-----------+----------------+---------+------------+----------------+--------------------+----------+



In [0]:
services_df.filter(col("service") == "Neurology").show()

+-----------+---------------+---------+----------+---------------+--------------------+---------+
|hospital_id|  hospital_name|     city|     phone|          email|            services|  service|
+-----------+---------------+---------+----------+---------------+--------------------+---------+
|       H101|Apollo Hospital|Hyderabad|9876500011|apollo@mail.com|[Cardiology, Neur...|Neurology|
|       H103|  Care Hospital|Bangalore|9876500013|   Not Provided|[Neurology, Pedia...|Neurology|
+-----------+---------------+---------+----------+---------------+--------------------+---------+



In [0]:
services_df.filter(col("service") == "Orthopedics").show()

+-----------+----------------+---------+------------+----------------+--------------------+-----------+
|hospital_id|   hospital_name|     city|       phone|           email|            services|    service|
+-----------+----------------+---------+------------+----------------+--------------------+-----------+
|       H102|Yashoda Hospital|Hyderabad|Not Provided|yashoda@mail.com|[Cardiology, Orth...|Orthopedics|
+-----------+----------------+---------+------------+----------------+--------------------+-----------+



In [0]:
services_df.filter(col("service") == "Pediatrics").show()

+-----------+-------------+---------+----------+------------+--------------------+----------+
|hospital_id|hospital_name|     city|     phone|       email|            services|   service|
+-----------+-------------+---------+----------+------------+--------------------+----------+
|       H103|Care Hospital|Bangalore|9876500013|Not Provided|[Neurology, Pedia...|Pediatrics|
+-----------+-------------+---------+----------+------------+--------------------+----------+



In [0]:
flat_df.write.mode("overwrite").parquet(
    "hospital_flattened.parquet"
)

In [0]:
flat_df.withColumn("services", concat_ws(", ", col("services"))).write.mode("overwrite").option(
    "header",
    True
).csv("hospital_flattened.csv")

In [0]:
doctor_revenue = doctor_visit_df.groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city").agg(
    sum("bill_amount").alias("total_revenue")
)

window = Window.orderBy(col("total_revenue").desc())

doctor_revenue.withColumn(
    "rank",
    rank().over(window)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+----+
|doctor_id|doctor_name|specialization|     city|total_revenue|rank|
+---------+-----------+--------------+---------+-------------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|   6|
+---------+-----------+--------------+---------+-------------+----+



In [0]:
doctor_revenue.withColumn(
    "dense_rank",dense_rank().over(window)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+----------+
|doctor_id|doctor_name|specialization|     city|total_revenue|dense_rank|
+---------+-----------+--------------+---------+-------------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|         6|
+---------+-----------+--------------+---------+-------------+----------+



In [0]:
doctor_revenue.withColumn(
    "row_number",row_number().over(window)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+----------+
|doctor_id|doctor_name|specialization|     city|total_revenue|row_number|
+---------+-----------+--------------+---------+-------------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|         6|
+---------+-----------+--------------+---------+-------------+----------+



In [0]:
doctor_revenue.withColumn(
    "rank",rank().over(window)
).filter(col("rank")==1).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+------+-------------+----+
|doctor_id|doctor_name|specialization|  city|total_revenue|rank|
+---------+-----------+--------------+------+-------------+----+
|     D104| Dr. Suresh|   Orthopedics|Mumbai|        18000|   1|
+---------+-----------+--------------+------+-------------+----+



In [0]:
doctor_revenue.withColumn(
    "rank",rank().over(window)
).filter(col("rank")<=3).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+----+
|doctor_id|doctor_name|specialization|     city|total_revenue|rank|
+---------+-----------+--------------+---------+-------------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|   3|
+---------+-----------+--------------+---------+-------------+----+



In [0]:
window = Window.partitionBy("specialization").orderBy(col("total_revenue").desc())

doctor_revenue.withColumn(
    "rank",rank().over(window)
).filter(
    col("rank")==1
).show()

+---------+-----------+--------------+---------+-------------+----+
|doctor_id|doctor_name|specialization|     city|total_revenue|rank|
+---------+-----------+--------------+---------+-------------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|   1|
+---------+-----------+--------------+---------+-------------+----+



In [0]:
doctor_revenue.withColumn(
    "rank",rank().over(window)
).filter(col("rank")<=2).show()

+---------+-----------+--------------+---------+-------------+----+
|doctor_id|doctor_name|specialization|     city|total_revenue|rank|
+---------+-----------+--------------+---------+-------------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|   2|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|   1|
+---------+-----------+--------------+---------+-------------+----+



In [0]:
window = Window.orderBy(col("total_revenue"))

doctor_revenue.withColumn(
    "running_revenue",sum("total_revenue").over(window)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+---------------+
|doctor_id|doctor_name|specialization|     city|total_revenue|running_revenue|
+---------+-----------+--------------+---------+-------------+---------------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|           1500|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|           3500|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|           7000|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|          14000|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|          24500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|          42500|
+---------+-----------+--------------+---------+-------------+---------------+



In [0]:
doctor_revenue.withColumn(
    "previous_revenue",lag("total_revenue").over(window)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+----------------+
|doctor_id|doctor_name|specialization|     city|total_revenue|previous_revenue|
+---------+-----------+--------------+---------+-------------+----------------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|            NULL|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|            1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|            2000|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|            3500|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|            7000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|           10500|
+---------+-----------+--------------+---------+-------------+----------------+



In [0]:
doctor_revenue.withColumn(
    "next_revenue",lead("total_revenue").over(window)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+------------+
|doctor_id|doctor_name|specialization|     city|total_revenue|next_revenue|
+---------+-----------+--------------+---------+-------------+------------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|        2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|        3500|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|        7000|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|       10500|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|       18000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|        NULL|
+---------+-----------+--------------+---------+-------------+------------+



In [0]:
doctor_revenue.withColumn(
    "previous_revenue",
    lag("total_revenue").over(window)
).withColumn(
    "comparison",
    when(col("total_revenue") >col("previous_revenue"),"Higher")
    .when(col("total_revenue")< col("previous_revenue"),"Lower")
    .otherwise("Same")
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+----------------+----------+
|doctor_id|doctor_name|specialization|     city|total_revenue|previous_revenue|comparison|
+---------+-----------+--------------+---------+-------------+----------------+----------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|            NULL|      Same|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|            1500|    Higher|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|            2000|    Higher|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|            3500|    Higher|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|            7000|    Higher|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|           10500|    Higher|
+---------+-----------+--------------+---------+-------------+----------------+----------+



In [0]:
doctor_revenue.withColumn(
    "next_revenue",
    lead("total_revenue").over(window)
).withColumn(
    "comparison",
    when(col("total_revenue") > col("next_revenue"),"Higher")
    .when(col("total_revenue") < col("next_revenue"),"Lower")
    .otherwise("Same")
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+------------+----------+
|doctor_id|doctor_name|specialization|     city|total_revenue|next_revenue|comparison|
+---------+-----------+--------------+---------+-------------+------------+----------+
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|        2000|     Lower|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|        3500|     Lower|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|        7000|     Lower|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|       10500|     Lower|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|       18000|     Lower|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|        NULL|      Same|
+---------+-----------+--------------+---------+-------------+------------+----------+



In [0]:
window = Window.partitionBy("city").orderBy(col("total_revenue").desc())

doctor_revenue.withColumn(
    "rank",rank().over(window)
).filter(
    col("rank")==1
).show()

+---------+-----------+--------------+---------+-------------+----+
|doctor_id|doctor_name|specialization|     city|total_revenue|rank|
+---------+-----------+--------------+---------+-------------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|   1|
+---------+-----------+--------------+---------+-------------+----+



In [0]:
window = Window.partitionBy("city").orderBy(col("total_revenue"))

doctor_revenue.withColumn(
    "rank",rank().over(window)).filter(
    col("rank")==1
).show()

+---------+-----------+--------------+---------+-------------+----+
|doctor_id|doctor_name|specialization|     city|total_revenue|rank|
+---------+-----------+--------------+---------+-------------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|   1|
+---------+-----------+--------------+---------+-------------+----+



In [0]:
window = Window.orderBy(col("total_revenue").desc())

doctor_revenue.withColumn(
    "rank",rank().over(window)
).select(
    "doctor_name",
    "specialization",
    "city",
    "total_revenue",
    "rank"
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------------+---------+-------------+----+
|doctor_name|specialization|     city|total_revenue|rank|
+-----------+--------------+---------+-------------+----+
| Dr. Suresh|   Orthopedics|   Mumbai|        18000|   1|
| Dr. Ramesh|    Cardiology|Hyderabad|        10500|   2|
|  Dr. Kiran|    Cardiology|Hyderabad|         7000|   3|
|  Dr. Priya|     Neurology|Bangalore|         3500|   4|
|  Dr. Anita|   Dermatology|  Chennai|         2000|   5|
|  Dr. Meera|    Pediatrics|    Delhi|         1500|   6|
+-----------+--------------+---------+-------------+----+



In [0]:
doc_df.createOrReplaceTempView("doctors")

In [0]:
visit_df.createOrReplaceTempView("visits")

In [0]:
services_df.createOrReplaceTempView("hospitals")

In [0]:
spark.sql("""
SELECT * FROM doctors
""").show()

+---------+-----------+--------------+---------+----------------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|
+---------+-----------+--------------+---------+----------------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|          2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|          1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|          3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|          1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|          1100|
+---------+-----------+--------------+---------+----------------+--------------+



In [0]:
spark.sql("""
SELECT specialization,COUNT(*) total_doctors FROM doctors
GROUP BY specialization
""").show()

+--------------+-------------+
|specialization|total_doctors|
+--------------+-------------+
|   Orthopedics|            1|
|    Cardiology|            2|
|    Pediatrics|            1|
|   Dermatology|            2|
|     Neurology|            2|
+--------------+-------------+



In [0]:
spark.sql("""
SELECT city, COUNT(*) total_doctors FROM doctors
GROUP BY city
""").show()

+---------+-------------+
|     city|total_doctors|
+---------+-------------+
|    Delhi|            1|
|  Chennai|            1|
|    Kochi|            1|
|Hyderabad|            2|
|Bangalore|            1|
|     Pune|            1|
|   Mumbai|            1|
+---------+-------------+



In [0]:
spark.sql("""
SELECT doctor_name,SUM(bill_amount) total_revenue FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY doctor_name
""").show()

+-----------+-------------+
|doctor_name|total_revenue|
+-----------+-------------+
| Dr. Ramesh|        10500|
|  Dr. Meera|         1500|
| Dr. Suresh|        18000|
|  Dr. Anita|         2000|
|  Dr. Kiran|         7000|
|  Dr. Priya|         3500|
+-----------+-------------+



In [0]:
spark.sql("""
SELECT specialization,SUM(bill_amount) total_revenue FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY specialization
""").show()

+--------------+-------------+
|specialization|total_revenue|
+--------------+-------------+
|   Orthopedics|        18000|
|    Cardiology|        17500|
|    Pediatrics|         1500|
|   Dermatology|         2000|
|     Neurology|         3500|
+--------------+-------------+



In [0]:
spark.sql("""
SELECT doctor_name,SUM(bill_amount) total_revenue FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY doctor_name
ORDER BY total_revenue DESC
LIMIT 5
""").show()

+-----------+-------------+
|doctor_name|total_revenue|
+-----------+-------------+
| Dr. Suresh|        18000|
| Dr. Ramesh|        10500|
|  Dr. Kiran|         7000|
|  Dr. Priya|         3500|
|  Dr. Anita|         2000|
+-----------+-------------+



In [0]:
spark.sql("""
SELECT * FROM visits
WHERE payment_='Pending'
""").show()

+--------+------------+---------+----------+------------+-----------+--------+-----+----------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_|  tax|final_bill|
+--------+------------+---------+----------+------------+-----------+--------+-----+----------+
|   V1003|  Amit Kumar|     D103|2026-01-11|Skin Allergy|       2000| Pending|100.0|    2100.0|
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|          0| Pending|  0.0|       0.0|
+--------+------------+---------+----------+------------+-----------+--------+-----+----------+



In [0]:
spark.sql("""
SELECT * FROM hospitals
WHERE service='Cardiology'
""").show()

+-----------+----------------+---------+------------+----------------+--------------------+----------+
|hospital_id|   hospital_name|     city|       phone|           email|            services|   service|
+-----------+----------------+---------+------------+----------------+--------------------+----------+
|       H101| Apollo Hospital|Hyderabad|  9876500011| apollo@mail.com|[Cardiology, Neur...|Cardiology|
|       H102|Yashoda Hospital|Hyderabad|Not Provided|yashoda@mail.com|[Cardiology, Orth...|Cardiology|
+-----------+----------------+---------+------------+----------------+--------------------+----------+



In [0]:
spark.sql("""
SELECT * FROM hospitals
WHERE service='Neurology'
""").show()

+-----------+---------------+---------+----------+---------------+--------------------+---------+
|hospital_id|  hospital_name|     city|     phone|          email|            services|  service|
+-----------+---------------+---------+----------+---------------+--------------------+---------+
|       H101|Apollo Hospital|Hyderabad|9876500011|apollo@mail.com|[Cardiology, Neur...|Neurology|
|       H103|  Care Hospital|Bangalore|9876500013|   Not Provided|[Neurology, Pedia...|Neurology|
+-----------+---------------+---------+----------+---------------+--------------------+---------+



In [0]:
spark.sql("""
SELECT * FROM hospitals
WHERE phone='Not Provided' OR email='Not Provided'
""").show()

+-----------+----------------+---------+------------+----------------+--------------------+-----------+
|hospital_id|   hospital_name|     city|       phone|           email|            services|    service|
+-----------+----------------+---------+------------+----------------+--------------------+-----------+
|       H102|Yashoda Hospital|Hyderabad|Not Provided|yashoda@mail.com|[Cardiology, Orth...| Cardiology|
|       H102|Yashoda Hospital|Hyderabad|Not Provided|yashoda@mail.com|[Cardiology, Orth...|Orthopedics|
|       H103|   Care Hospital|Bangalore|  9876500013|    Not Provided|[Neurology, Pedia...|  Neurology|
|       H103|   Care Hospital|Bangalore|  9876500013|    Not Provided|[Neurology, Pedia...| Pediatrics|
+-----------+----------------+---------+------------+----------------+--------------------+-----------+



In [0]:
spark.sql("""
SELECT AVG(consultation_f) average_fee FROM doctors
""").show()

+-----------+
|average_fee|
+-----------+
|     1762.5|
+-----------+



In [0]:
spark.sql("""
SELECT doctor_name,specialization,city,
COUNT(visit_id) total_visits,
SUM(bill_amount) total_revenue,
AVG(bill_amount) average_bill
FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY doctor_name,specialization,city
ORDER BY total_revenue DESC
""").show()

+-----------+--------------+---------+------------+-------------+------------+
|doctor_name|specialization|     city|total_visits|total_revenue|average_bill|
+-----------+--------------+---------+------------+-------------+------------+
| Dr. Suresh|   Orthopedics|   Mumbai|           2|        18000|      9000.0|
| Dr. Ramesh|    Cardiology|Hyderabad|           2|        10500|      5250.0|
|  Dr. Kiran|    Cardiology|Hyderabad|           1|         7000|      7000.0|
|  Dr. Priya|     Neurology|Bangalore|           1|         3500|      3500.0|
|  Dr. Anita|   Dermatology|  Chennai|           2|         2000|      1000.0|
|  Dr. Meera|    Pediatrics|    Delhi|           1|         1500|      1500.0|
+-----------+--------------+---------+------------+-------------+------------+



In [0]:
doctor_df = doc_df
visits_df = visit_df
hospital_df = flat_df

In [0]:
visit_df = visit_df.fillna({
    "bill_amount":0
})

hospital_df = hospital_df.fillna({
    "phone":"Not Provided",
    "email":"Not Provided"
})

In [0]:
hospital_df.show()

+-----------+----------------+---------+------------+----------------+--------------------+
|hospital_id|   hospital_name|     city|       phone|           email|            services|
+-----------+----------------+---------+------------+----------------+--------------------+
|       H101| Apollo Hospital|Hyderabad|  9876500011| apollo@mail.com|[Cardiology, Neur...|
|       H102|Yashoda Hospital|Hyderabad|Not Provided|yashoda@mail.com|[Cardiology, Orth...|
|       H103|   Care Hospital|Bangalore|  9876500013|    Not Provided|[Neurology, Pedia...|
+-----------+----------------+---------+------------+----------------+--------------------+



In [0]:
hospital_etl_df = doctor_df.join(
    visits_df,"doctor_id","left"
)

hospital_etl_df.show()

+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_f|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+--------------+--------+------------+----------+-------------+-----------+--------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|          1500|   V1010| Nisha Reddy|2026-01-14|Heart Checkup|       5500|    Paid|275.0|    5775.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|          2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|    Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|          1000|   V1008|  Meera Nair|2026-01-13| Skin Allergy|          0| Pending|  0.0|       0.0|
|   

In [0]:
doctor_rank_df = hospital_etl_df.groupBy(
    "doctor_name").agg(
    sum("bill_amount").alias("total_revenue")
)

window = Window.orderBy(col("total_revenue").desc())

doctor_rank_df.withColumn(
    "rank",rank().over(window)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+-------------+----+
|doctor_name|total_revenue|rank|
+-----------+-------------+----+
| Dr. Suresh|        18000|   1|
| Dr. Ramesh|        10500|   2|
|  Dr. Kiran|         7000|   3|
|  Dr. Priya|         3500|   4|
|  Dr. Anita|         2000|   5|
|  Dr. Meera|         1500|   6|
|  Dr. Nisha|         NULL|   7|
| Dr. Farhan|         NULL|   7|
+-----------+-------------+----+



In [0]:
hospital_etl_df.groupBy(
    "specialization"
).agg(
    count("visit_id").alias("total_visits"),
    sum("bill_amount").alias("total_revenue"),
    avg("bill_amount").alias("average_bill")
).show()

+--------------+------------+-------------+-----------------+
|specialization|total_visits|total_revenue|     average_bill|
+--------------+------------+-------------+-----------------+
|   Orthopedics|           2|        18000|           9000.0|
|    Cardiology|           3|        17500|5833.333333333333|
|    Pediatrics|           1|         1500|           1500.0|
|   Dermatology|           2|         2000|           1000.0|
|     Neurology|           1|         3500|           3500.0|
+--------------+------------+-------------+-----------------+



In [0]:
hospital_etl_df.write.mode("overwrite").parquet(
    "silver/hospital_data"
)

In [0]:
gold_report = hospital_etl_df.groupBy(
    "doctor_name",
    "specialization",
    "city"
).agg(
    count("visit_id").alias("total_visits"),
    sum("bill_amount").alias("total_revenue"),
    avg("bill_amount").alias("average_bill")
)

gold_report.show()

gold_report.write.mode("overwrite").parquet(
    "gold/doctor_report"
)

+-----------+--------------+---------+------------+-------------+------------+
|doctor_name|specialization|     city|total_visits|total_revenue|average_bill|
+-----------+--------------+---------+------------+-------------+------------+
|  Dr. Anita|   Dermatology|  Chennai|           2|         2000|      1000.0|
|  Dr. Kiran|    Cardiology|Hyderabad|           1|         7000|      7000.0|
|  Dr. Meera|    Pediatrics|    Delhi|           1|         1500|      1500.0|
|  Dr. Nisha|   Dermatology|    Kochi|           0|         NULL|        NULL|
| Dr. Suresh|   Orthopedics|   Mumbai|           2|        18000|      9000.0|
| Dr. Farhan|     Neurology|     Pune|           0|         NULL|        NULL|
| Dr. Ramesh|    Cardiology|Hyderabad|           2|        10500|      5250.0|
|  Dr. Priya|     Neurology|Bangalore|           1|         3500|      3500.0|
+-----------+--------------+---------+------------+-------------+------------+



In [0]:
dashboard_df = hospital_etl_df.select(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city",
    "patient_name",
    "diagnosis",
    "bill_amount",
    "tax",
    "final_bill",
    "payment_"
)

dashboard_df.show()

dashboard_df.write.mode("overwrite").parquet(
    "gold/hospital_dashboard")

+---------+-----------+--------------+---------+------------+-------------+-----------+-----+----------+--------+
|doctor_id|doctor_name|specialization|     city|patient_name|    diagnosis|bill_amount|  tax|final_bill|payment_|
+---------+-----------+--------------+---------+------------+-------------+-----------+-----+----------+--------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad| Nisha Reddy|Heart Checkup|       5500|275.0|    5775.0|    Paid|
|     D102|  Dr. Priya|     Neurology|Bangalore| Priya Reddy|     Migraine|       3500|175.0|    3675.0|    Paid|
|     D103|  Dr. Anita|   Dermatology|  Chennai|  Meera Nair| Skin Allergy|          0|  0.0|       0.0| Pending|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|   Kiran Rao|    Back Pain|       6000|300.0|    6300.0|    Paid|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|  Farhan Ali|        Fever|       1500| 75.0|    1575.0|    Paid|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|  Neha Singh|Heart Checkup|       7000|3